# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

This dataset includes survey data on mental health indicators among residents of Kilifi County, Kenya. It contains demographic information and psychological symptom scores from validated scales such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Use the `@id` field to reference entities.

In [ ]:
# Explore record sets in the metadata
record_sets = dataset.metadata.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}\n      Name: {field.name}\n      Data type: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - @id: {col.id}\n      Name: {col.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For illustration, we'll extract all record sets and their fields using their `@id`.

In [ ]:
# List record set IDs
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for Record Set @id '{record_set_id}': {df.columns.tolist()}")
    print(df.head(2))
    print("---\n")
# Let's select the first record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll use the record set and field `@id`s from the above extraction.

In [ ]:
# Select a numeric field for analysis
# Identify numeric fields (e.g. GAD-7, PHQ-9, PSQ scores)
df = dataframes[main_record_set_id]
numeric_fields = [col for col in df.columns if 'score' in col.lower() or col.lower() in ['phq9_score', 'gad7_score', 'psq_score']]

# If none found, try a default
selected_numeric_field = numeric_fields[0] if numeric_fields else df.columns[0]

threshold = 10
if selected_numeric_field in df.columns:
    filtered_df = df[df[selected_numeric_field] > threshold]
    print(f"Filtered records with {selected_numeric_field} > {threshold}:")
    print(filtered_df.head(3))

    # Normalize numeric field
    filtered_df[f"{selected_numeric_field}_normalized"] = (
        filtered_df[selected_numeric_field] - filtered_df[selected_numeric_field].mean()
    ) / filtered_df[selected_numeric_field].std()
    print(f"Normalized {selected_numeric_field} for filtered records:")
    print(filtered_df[[selected_numeric_field, f"{selected_numeric_field}_normalized"]].head())

    # Group by a categorical field (e.g. gender or education)
    possible_groups = ['gender', 'level_of_education', 'village', 'marital_status']
    group_field = next((g for g in possible_groups if g in filtered_df.columns), None)

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[selected_numeric_field].mean().reset_index()
        print(f"Grouped means of {selected_numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[selected_numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {selected_numeric_field}")
plt.xlabel(selected_numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field was set, plot groupwise means
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(8,5))
    sns.barplot(x=group_field, y=selected_numeric_field, data=grouped_df)
    plt.title(f"Mean {selected_numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {selected_numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich survey data on mental health, with demographic and psychological indicator fields.
- We filtered respondents with high scores on the selected indicator and examined group differences by demographic categories.
- Visualization shows the distribution and groupwise mean variation for the chosen mental health metric.
- This approach demonstrates using Croissant metadata (`@id` referencing for record sets and fields) to access and process dataset contents in a reproducible, FAIR-enabled manner.